In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw data
df = pd.read_csv('../data/train.csv')

# See all columns with missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"Columns with missing values: {len(missing)}")
print(missing)

Columns with missing values: 19
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
MasVnrArea         8
Electrical         1
dtype: int64


In [2]:
# ── Group 1: NaN means "no feature" ──────────────────────────
# Text columns → fill with "None"
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtFinType1', 'BsmtFinType2', 'BsmtExposure',
             'BsmtCond', 'BsmtQual', 'MasVnrType']

for col in none_cols:
    df[col] = df[col].fillna('None')

# Numeric columns → fill with 0
zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
             'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in zero_cols:
    df[col] = df[col].fillna(0)

# ── Group 2: Genuinely missing ────────────────────────────────
# LotFrontage → fill with median per neighbourhood
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median()))

# Electrical → fill with most common value
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

# ── Check: should be 0 missing now ───────────────────────────
print("Missing values remaining:", df.isnull().sum().sum())

Missing values remaining: 0


In [3]:
# See which columns are still text
cat_cols = df.select_dtypes(include=['object']).columns
print(f"Categorical columns to encode: {len(cat_cols)}")
print(cat_cols.tolist())

Categorical columns to encode: 43
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


/var/folders/4s/nsmjrmn96tjf47kyt5dvy9800000gn/T/ipykernel_31082/1900510345.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [4]:
# Convert all text columns to numbers using one-hot encoding
df = pd.get_dummies(df, columns=list(cat_cols))
print("Shape after encoding:", df.shape)
print("All columns are now numeric:", df.select_dtypes(include=['object']).empty)

Shape after encoding: (1460, 304)
All columns are now numeric: True


In [5]:
# Remove outliers — huge houses sold cheap confuse the model
# Drop houses with GrLivArea > 4000 sqft but sold cheap (known outliers)
df = df.drop(df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)].index)

print("Shape after removing outliers:", df.shape)

Shape after removing outliers: (1458, 304)


In [6]:
# Save cleaned data so we can use it in the model notebook
df.to_csv('../data/clean.csv', index=False)
print("✅ Clean data saved to data/clean.csv")
print(f"Final shape: {df.shape}")


✅ Clean data saved to data/clean.csv
Final shape: (1458, 304)
